# Sionna 0.19 – Differentiable Ray Tracing: Material Calibration & TX Optimization
**Kernel:** `sionna019` · Python 3.10 · Sionna 0.19.2 · TensorFlow 2.15

Fixed from original sionna_019_differentiable_rt.ipynb:
- `load_scene()` — removed invalid `merge_shapes=True` argument (not supported in Sionna 0.19)
- `UTM_EPSG` — corrected from 32631 (Paris/zone-31N) to 32630 (UK/zone-30N) for London scenes
- DEM CRS auto-detection (BNG vs WGS84) for correct coordinate lookup

Implements NVlabs/diff-rt pattern: NMSE OFDM loss · Adam · `check_mat()` · TX orientation optimization.

## CELL 0 · Environment Setup & Imports

In [ ]:
import os, sys, json, csv, time, warnings, importlib
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from pyproj import Transformer

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver

# ── Mitsuba (ray casting) ──────────────────────────────────────────────────────
_HAS_MI = False
try:
    import mitsuba as mi
    try:   mi.set_variant('cuda_ad_mono_polarized')
    except: mi.set_variant('llvm_ad_mono_polarized')
    _HAS_MI = True
    print(f'Mitsuba : {mi.variant()}')
except ImportError:
    print('Mitsuba : NOT available – ray-cast ground height disabled')

# ── rasterio (DEM lookup) ──────────────────────────────────────────────────────
_HAS_RIO = False
try:
    import rasterio as rio
    _HAS_RIO = True
except ImportError:
    print('rasterio: NOT available – DEM elevation disabled')

# ── OFDM helpers (diff-rt NMSE loss) ──────────────────────────────────────────
_HAS_OFDM = False
for _pkg in ('sionna.channel', 'sionna.channel.ofdm'):
    try:
        _m = importlib.import_module(_pkg)
        cir_to_ofdm_channel    = _m.cir_to_ofdm_channel
        subcarrier_frequencies = _m.subcarrier_frequencies
        _HAS_OFDM = True
        print(f'OFDM    : OK  ({_pkg})')
        break
    except (ImportError, AttributeError):
        continue
if not _HAS_OFDM:
    print('OFDM    : NOT found – power-domain fallback will be used')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')
print(f'GPU(s)  : {[g.name for g in tf.config.list_physical_devices("GPU")]}')

# ── Shared helpers ─────────────────────────────────────────────────────────────
def _safe(v):
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')

## CELL 1 · Project & Scene Detection

Auto-detects scene.xml from the project folder.
Reads scene bounds from XML `<default>` tags.

In [ ]:
PROJECT_ROOT = '/home/georgeskai/Documents/Region/london3602/londonflat_scene3602'
if not os.path.isdir(PROJECT_ROOT):
    PROJECT_ROOT = '/home/georgeskai/Documents/Region/london3602/londonflat_scene3602-day1'

candidates = sorted(
    [d for d in os.listdir(PROJECT_ROOT)
     if os.path.isdir(os.path.join(PROJECT_ROOT, d))],
    key=lambda d: os.path.getmtime(os.path.join(PROJECT_ROOT, d)), reverse=True)

PROJECT_NAME = ''; PROJECT_PATH = None; SCENE_XML = None; DEM_TIFF = None

for name in candidates:
    path = os.path.join(PROJECT_ROOT, name)
    candidate_xml = os.path.join(path, 'scene', 'scene.xml')
    if os.path.exists(candidate_xml):
        PROJECT_NAME = name; PROJECT_PATH = path
        SCENE_XML = candidate_xml
        DEM_TIFF  = os.path.join(path, 'scene', 'dem_wgs84.tif')
        break

if not PROJECT_NAME:
    direct_xml = os.path.join(PROJECT_ROOT, 'scene.xml')
    if os.path.exists(direct_xml):
        PROJECT_NAME = os.path.basename(PROJECT_ROOT)
        PROJECT_PATH = PROJECT_ROOT
        SCENE_XML    = direct_xml
        DEM_TIFF     = os.path.join(PROJECT_ROOT, 'dem_wgs84.tif')
        print(f'No project subfolder; using direct scene.xml at {SCENE_XML}')
    else:
        raise FileNotFoundError(f'No scene.xml found in {PROJECT_ROOT} or its subdirectories.')

print(f'Project : {PROJECT_NAME}')
print(f'Path    : {PROJECT_PATH}')
print(f'Scene   : {SCENE_XML}')
print(f'DEM     : {DEM_TIFF}')

_pjson = os.path.join(PROJECT_PATH, 'project.json')
_PROJ  = {}
if os.path.exists(_pjson):
    with open(_pjson) as f:
        _PROJ = json.load(f)

def get_xml_default(root, name, fallback=None):
    elem = root.find(f"./default[@name='{name}']")
    return float(elem.get('value')) if elem is not None else fallback

XML_OK = os.path.exists(SCENE_XML)
WEST = EAST = SOUTH = NORTH = center_lon = center_lat = None

if XML_OK:
    try:
        _root = ET.parse(SCENE_XML).getroot()
        _ver  = _root.get('version', 'unknown')
        _maj  = int(_ver.split('.')[0]) if _ver and _ver[0].isdigit() else 0
        print(f'XML version : {_ver}  →  {"✓ COMPATIBLE (Mitsuba 3)" if _maj >= 3 else "✗ INCOMPATIBLE (Mitsuba 2)"}')
        WEST       = get_xml_default(_root, 'scenegen_min_lon')
        EAST       = get_xml_default(_root, 'scenegen_max_lon')
        SOUTH      = get_xml_default(_root, 'scenegen_min_lat')
        NORTH      = get_xml_default(_root, 'scenegen_max_lat')
        center_lon = get_xml_default(_root, 'scenegen_origin_lon')
        center_lat = get_xml_default(_root, 'scenegen_origin_lat')
    except ET.ParseError as e:
        print(f'XML parse error: {e}'); XML_OK = False; _root = None
else:
    print('✗ scene.xml NOT FOUND'); _root = None

if None in [WEST, EAST, SOUTH, NORTH]:
    _bb = _PROJ.get('bbox', {})
    WEST  = _bb.get('min_lon',  -0.150720)
    EAST  = _bb.get('max_lon',  -0.048410)
    SOUTH = _bb.get('min_lat',  51.497840)
    NORTH = _bb.get('max_lat',  51.560640)
    print('  Using project.json bbox as fallback.')

if center_lon is None: center_lon = (WEST  + EAST)  / 2
if center_lat is None: center_lat = (SOUTH + NORTH) / 2

print(f'Bounds : lon [{WEST:.6f}, {EAST:.6f}]  lat [{SOUTH:.6f}, {NORTH:.6f}]')
print(f'Center : ({center_lon:.6f}, {center_lat:.6f})')

## CELL 2 · Global Configuration

In [ ]:
OUTPUT_DIR = os.path.join(PROJECT_PATH, 'results', 'diff_rt')
os.makedirs(OUTPUT_DIR, exist_ok=True)

_ant         = _PROJ.get('antenna_config', {})
FREQUENCY_HZ = float(_ant.get('frequency_hz', 3.6e9))
BANDWIDTH_HZ = float(_ant.get('bandwidth_hz', 20e6))
TX_POWER_DBM = 43.0
TX_GAIN_DBI  = 0.0
RX_GAIN_DBI  = 0.0
LNA_GAIN_DB  = 0.0
NOISE_FLOOR  = -120.0
EIRP_DBM     = TX_POWER_DBM + TX_GAIN_DBI

# ── Coordinate system ─────────────────────────────────────────────────────────
# FIX: was 32631 (Paris/zone-31N) — corrected to 32630 for London (zone-30N)
# Verify: for center_lon ≈ -0.1 (London) → UTM zone 30N = EPSG:32630
UTM_EPSG = 32630

# ── CSV input files ───────────────────────────────────────────────────────────
TX_CSV = os.path.join(PROJECT_PATH, 'transmitter_positions.csv')
RX_CSV = os.path.join(PROJECT_PATH, 'receiver_locations1.csv')

# ── OFDM parameters ───────────────────────────────────────────────────────────
NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if _HAS_OFDM:
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

# ── Path solver parameters ────────────────────────────────────────────────────
_ps           = _ant.get('path_solver', {})
MAX_DEPTH     = int(_ps.get('max_depth', 5))
NUM_SAMPLES_CM  = 5_000_000
NUM_SAMPLES_PS  = 2_000_000
GRID_SIZE_M   = 5.0

# ── diff-rt material calibration parameters ───────────────────────────────────
CALIB_STEPS    = 100
CALIB_LR       = 0.1
CALIB_NUM_SAMP = 500_000
CALIB_DEPTH    = 3

# ── TX orientation optimization ───────────────────────────────────────────────
ORI_STEPS    = 30
ORI_LR       = 0.1
ORI_NUM_SAMP = 1_000_000

_tx_w    = 10**((TX_POWER_DBM - 30) / 10)
_noise_w = 10**((NOISE_FLOOR  - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

print('=' * 60)
print('SYSTEM CONFIGURATION')
print('=' * 60)
print(f'Output dir     : {OUTPUT_DIR}')
print(f'Frequency      : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'TX power       : {TX_POWER_DBM} dBm  →  EIRP = {EIRP_DBM} dBm')
print(f'Noise floor    : {NOISE_FLOOR} dBm')
print(f'SNR scale      : {SNR_SCALE:.2e}')
print(f'UTM EPSG       : {UTM_EPSG}  (London = zone-30N)')
print(f'Max depth      : {MAX_DEPTH}')
print(f'Calib steps    : {CALIB_STEPS}  LR={CALIB_LR}')
print(f'Ori steps      : {ORI_STEPS}    LR={ORI_LR}')
print('=' * 60)

## CELL 3 · Coordinate Utilities + DEM Elevation

- `gps_to_local(lon, lat)` → UTM → subtract scene origin → local XY
- `local_to_gps(x, y)` → reverse
- `get_dem_elevation(local_x, local_y)` → rasterio bilinear on `dem_wgs84.tif`
- `ray_cast_ground_z(x, y)` → Mitsuba ray intersect for terrain height

In [ ]:
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    """GPS (lon, lat) → Sionna local XY (metres from scene origin)."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Sionna local XY → GPS (lon, lat)."""
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

# ── DEM bilinear lookup ───────────────────────────────────────────────────────
dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())
if _is_bng_dem:
    utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)

def get_dem_elevation(local_x, local_y):
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    if _is_bng_dem:
        px, py = utm_to_bng.transform(utm_x, utm_y)
    else:
        px, py = utm_to_gps.transform(utm_x, utm_y)  # WGS84 lon/lat
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if not _HAS_MI: return get_dem_elevation(x, y)
    try:
        ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                       mi.Vector3f(0.0, 0.0, -1.0))
        si = scene.mi_scene.ray_intersect(ray)
        if si.is_valid():
            z_val = si.p.z
            return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
    except Exception:
        pass
    return get_dem_elevation(x, y)

print('Coordinate utilities ready.')
print(f'  gps_to_local({center_lon:.4f}, {center_lat:.4f}) → {gps_to_local(center_lon, center_lat)[:2]}')

## CELL 4 · Load 3-D Scene & Configure Antennas

**FIX:** Removed `merge_shapes=True` — this argument does not exist in Sionna 0.19's `load_scene()`.
The original error was: `TypeError: load_scene() got an unexpected keyword argument 'merge_shapes'`

In [ ]:
if not XML_OK:
    raise RuntimeError(
        'scene.xml not found. Complete Steps 1-3 in the sionna_web UI:\n'
        '  1. Draw area on map\n'
        '  2. Configure materials\n'
        '  3. Click "Generate 3-D Scene"')

print(f'Loading scene from {SCENE_XML} ...')
# FIX: load_scene() in Sionna 0.19 does NOT accept merge_shapes=True
scene = load_scene(SCENE_XML)
scene.frequency = FREQUENCY_HZ

def _make_array(cfg):
    return PlanarArray(
        num_rows           = cfg.get('num_rows',           1),
        num_cols           = cfg.get('num_cols',           1),
        vertical_spacing   = cfg.get('vertical_spacing',   0.5),
        horizontal_spacing = cfg.get('horizontal_spacing', 0.5),
        pattern            = cfg.get('pattern',            'iso'),
        polarization       = cfg.get('polarization',       'V'),
    )

scene.tx_array = _make_array(_ant.get('tx_array', {}))
scene.rx_array = _make_array(_ant.get('rx_array', {}))

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')
print(f'TX array      : {scene.tx_array}')

## CELL 4A · Assign ITU-R P.2040-2 Material Properties

In [ ]:
_ITU_DB = {
    'concrete'          : (5.24,  0.130, 0.40, 0.20),
    'brick'             : (3.91,  0.024, 0.30, 0.20),
    'wood'              : (1.99,  0.005, 0.25, 0.30),
    'glass'             : (6.27,  0.012, 0.08, 0.10),
    'metal'             : (1.00,  1e7,   0.05, 0.10),
    'asphalt'           : (3.00,  0.010, 0.35, 0.20),
    'vegetation'        : (1.30,  0.001, 0.75, 0.05),
    'water'             : (81.0,  0.500, 0.02, 0.05),
    'wet_ground'        : (30.0,  0.150, 0.20, 0.20),
    'medium_dry_ground' : (15.0,  0.035, 0.18, 0.20),
    'very_dry_ground'   : (3.00,  0.001, 0.12, 0.20),
    'marble'            : (7.07,  0.020, 0.08, 0.10),
    'plasterboard'      : (2.73,  0.010, 0.12, 0.20),
    'chipboard'         : (2.58,  0.012, 0.14, 0.20),
    'plywood'           : (2.71,  0.014, 0.15, 0.20),
    'ceiling_board'     : (1.50,  0.006, 0.13, 0.20),
    'floorboard'        : (2.00,  0.010, 0.16, 0.20),
}
_DEFAULT_MAT = (4.0, 0.08, 0.30, 0.15)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    for key in _ITU_DB:
        if key in n: return key
    for key in _ITU_DB:
        if any(part in n for part in key.split('_')): return key
    return None

_SCATTER_CFG = {
    'concrete': (0.15, 0.30), 'brick': (0.10, 0.25), 'wood': (0.10, 0.10),
    'glass': (0.05, 0.01), 'metal': (0.05, 0.001), 'wet_ground': (0.05, 0.10),
    'medium_dry_ground': (0.05, 0.10), 'very_dry_ground': (0.05, 0.10),
    'marble': (0.05, 0.30), 'plasterboard': (0.10, 0.02), 'chipboard': (0.10, 0.02),
    'plywood': (0.10, 0.02), 'ceiling_board': (0.10, 0.02), 'floorboard': (0.10, 0.02),
    'vegetation': (0.75, 1.00), 'asphalt': (0.35, 0.15), 'water': (0.02, 0.10),
}

print('=' * 70)
print('ASSIGNING ITU-R MATERIAL PROPERTIES  (auto-match by name)')
print('=' * 70)
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    eps_r, sigma, S, xpd = _ITU_DB.get(key, _DEFAULT_MAT)
    sc, th = _SCATTER_CFG.get(key, (0.20, 0.10))
    try: mat.relative_permittivity = eps_r
    except Exception: pass
    try: mat.conductivity = sigma
    except Exception: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, sc); break
            except: pass
    for a_ in ('xpd_coefficient', 'xpd_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, xpd); break
            except: pass
    try: mat.thickness = th
    except: pass
    print(f'  {mat_name:<32}  matched={key or "DEFAULT":<20}  '
          f'eps={eps_r:.2f}  σ={sigma:.4g}  S={sc:.2f}')
print('Done.')

## CELL 6 · Load Transmitter (GPS → local XY, ray-cast Z)

In [ ]:
print('=' * 70)
print('CELL 6 – LOAD TRANSMITTER')
print('=' * 70)

for nm in list(scene.transmitters.keys()):
    scene.remove(nm)

if os.path.exists(TX_CSV):
    df_tx    = pd.read_csv(TX_CSV)
    row      = df_tx.iloc[0]
    tx_name  = str(row.get('name', 'tx_0'))
    tx_lon   = float(row['lon'])
    tx_lat   = float(row['lat'])
    tx_agl   = float(row.get('height', 25.0))
    tx_power = float(row.get('power_dbm', EIRP_DBM))
    source   = 'CSV'
else:
    _tx_cfg  = _ant.get('transmitters', [{'name':'tx0','position':[0,0,25]}])[0]
    _pos     = _tx_cfg.get('position', [0, 0, 25])
    tx_name  = _tx_cfg.get('name', 'tx0')
    tx_lon, tx_lat = local_to_gps(_pos[0], _pos[1])
    tx_agl   = _pos[2]
    tx_power = EIRP_DBM
    source   = 'project.json'
    print(f'  TX CSV not found – using {source}')

print(f'[1] Source      : {source}')
print(f'    GPS         : ({tx_lon:.6f}, {tx_lat:.6f})  AGL={tx_agl:.1f} m')

local_x, local_y, _ = gps_to_local(tx_lon, tx_lat)
print(f'[2] Local XY    : ({local_x:.2f}, {local_y:.2f})')

ground_z = ray_cast_ground_z(local_x, local_y)
if ground_z == 0.0:
    try:
        _bbox    = scene.mi_scene.bbox()
        ground_z = float(_bbox.min[2])
    except: pass
abs_z = ground_z + tx_agl
print(f'[3] Ground Z    : {ground_z:.2f} m  +  AGL {tx_agl:.1f} m  →  abs Z={abs_z:.2f} m')

tx = Transmitter(name=tx_name,
                 position=(float(local_x), float(local_y), float(abs_z)),
                 power_dbm=float(tx_power))
scene.add(tx)
print(f'[4] ✓ Added TX "{tx_name}"  pos=({local_x:.1f}, {local_y:.1f}, {abs_z:.1f})  EIRP={tx_power:.1f} dBm')

## CELL 7 · Load Receivers (GPS → local XY, ray-cast Z)

In [ ]:
print('=' * 70)
print('CELL 7 – LOAD RECEIVERS')
print('=' * 70)

for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []

if os.path.exists(RX_CSV):
    df_rx = pd.read_csv(RX_CSV)
    print(f'[1] Loaded {len(df_rx)} receivers from {RX_CSV}')
    print('[2] Converting GPS → local XY + ray-cast ground Z ...')
    t0 = time.time()
    for i, row in df_rx.iterrows():
        lon  = float(row['lon'])
        lat  = float(row['lat'])
        agl  = float(row.get('height', 1.5))
        x, y, _ = gps_to_local(lon, lat)
        gz   = ray_cast_ground_z(x, y)
        z    = gz + agl
        nm   = str(row.get('name', f'RX_{i+1:04d}'))
        rx   = Receiver(name=nm, position=(float(x), float(y), float(z)))
        scene.add(rx)
        receivers.append(rx)
    print(f'    Done in {time.time()-t0:.2f} s')
else:
    print('  RX CSV not found – using project.json receivers')
    for rx_cfg in _ant.get('receivers', [{'name':'rx0','position':[100,0,1.5]}]):
        pos = rx_cfg.get('position', [100, 0, 1.5])
        nm  = rx_cfg.get('name', f'rx{len(receivers)}')
        rx  = Receiver(name=nm, position=(float(pos[0]), float(pos[1]), float(pos[2])))
        scene.add(rx)
        receivers.append(rx)

print(f'[3] {len(receivers)} receivers placed')
print('[4] First 5 receivers:')
for rx in receivers[:5]:
    x, y, z = _safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name:<15} XY=({x:8.1f}, {y:8.1f})  Z={z:.2f}  GPS=({lon:.5f}, {lat:.5f})')

try:
    _bbox = scene.mi_scene.bbox()
    ok_x  = all(float(_bbox.min[0]) <= _safe(r.position[0]) <= float(_bbox.max[0]) for r in receivers)
    ok_y  = all(float(_bbox.min[1]) <= _safe(r.position[1]) <= float(_bbox.max[1]) for r in receivers)
    print(f'[5] All inside scene bbox: X={ok_x}  Y={ok_y}')
except: pass

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')

## CELL 8 · Pre-Calibration Coverage Map

In [ ]:
print('Pre-calibration coverage map (ITU material defaults) ...')

try:
    _bbox = scene.mi_scene.bbox()
    cx = (float(_bbox.min[0]) + float(_bbox.max[0])) / 2
    cy = (float(_bbox.min[1]) + float(_bbox.max[1])) / 2
except Exception:
    cx = cy = 0.0

ground_z_centre = ray_cast_ground_z(cx, cy)
if ground_z_centre == 0.0:
    ground_z_centre = get_dem_elevation(cx, cy)
cm_height = ground_z_centre + 1.5
print(f'  Coverage map plane Z = {ground_z_centre:.2f} + 1.5 = {cm_height:.2f} m')

cm_pre = scene.coverage_map(
    cm_cell_size        = GRID_SIZE_M,
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_CM,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = False,
)
cm_pre_np = _cm_to_numpy(cm_pre)

pg_db = 10 * np.log10(cm_pre_np[0] + 1e-30)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(pg_db, origin='lower', cmap='jet',
               vmin=np.nanpercentile(pg_db, 5), vmax=np.nanpercentile(pg_db, 99))
plt.colorbar(im, ax=ax, label='Path Gain (dB)')
ax.set_title('Pre-Calibration Coverage Map – ITU Material Defaults')
ax.set_xlabel('X cells'); ax.set_ylabel('Y cells')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_pre_calibration.png'), dpi=150)
plt.show()

# ── Interpolate to RX positions via KDTree ────────────────────────────────────
H, W = pg_db.shape
try:
    _bbox = scene.mi_scene.bbox()
    _gx_min, _gx_max = float(_bbox.min[0]), float(_bbox.max[0])
    _gy_min, _gy_max = float(_bbox.min[1]), float(_bbox.max[1])
except Exception:
    _gx_min = _gy_min = -500.0; _gx_max = _gy_max = 500.0

x_centers = np.linspace(_gx_min, _gx_max, W)
y_centers  = np.linspace(_gy_min, _gy_max, H)
XX, YY = np.meshgrid(x_centers, y_centers)
tree   = KDTree(np.column_stack([XX.ravel(), YY.ravel()]))
rx_coords   = np.array([(_safe(rx.position[0]), _safe(rx.position[1])) for rx in receivers])
_, _indices = tree.query(rx_coords)
pg_at_rx_pre = pg_db.ravel()[_indices]

print(f'Pre-calibration path-gain at RX:  '
      f'mean={np.mean(pg_at_rx_pre):.1f} dB  '
      f'min={np.min(pg_at_rx_pre):.1f} dB  max={np.max(pg_at_rx_pre):.1f} dB')

## CELL 8b · Reference Channel – Ground-Truth OFDM Response

In [ ]:
def compute_h_freq(sc, num_samp=CALIB_NUM_SAMP, depth=CALIB_DEPTH):
    """
    Compute OFDM channel frequency response using Sionna 0.19 API.
    Returns tf.Tensor [num_rx, num_subcarriers] (complex) or [num_rx] (power fallback).
    """
    paths = sc.compute_paths(
        max_depth           = depth,
        num_samples         = num_samp,
        los                 = True,
        specular_reflection = True,
        diffuse_reflection  = True,
        refraction          = True,
        diffraction         = False,
    )
    if _HAS_OFDM:
        try:
            a, tau = paths.cir()
            h = cir_to_ofdm_channel(FREQUENCIES, a, tau, normalize=False)
            return tf.squeeze(h)
        except Exception as e:
            print(f'  cir() fallback: {e}')
    # Power fallback → [num_rx]
    a_t = paths.a
    if isinstance(a_t, tuple): a_t = tf.complex(a_t[0], a_t[1])
    if a_t.shape[0] == 1: a_t = a_t[0]
    return tf.cast(
        tf.reduce_sum(tf.abs(a_t)**2, axis=list(range(1, len(a_t.shape)))),
        tf.float32)

print('Computing reference channel (ITU defaults + full depth) ...')
print(f'  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')
h_ref    = compute_h_freq(scene, num_samp=NUM_SAMPLES_PS, depth=MAX_DEPTH)
h_ref_np = _to_numpy(h_ref)
print(f'Reference shape : {h_ref_np.shape}  dtype={h_ref_np.dtype}')
pwr_range = 10*np.log10(np.abs(h_ref_np)**2 + 1e-30)
print(f'Power range     : {pwr_range.min():.1f} … {pwr_range.max():.1f} dB')
h_ref_tf = tf.constant(h_ref_np,
                        dtype=tf.complex64 if np.iscomplexobj(h_ref_np) else tf.float32)
print(f'Stored as       : tf.constant {h_ref_tf.shape}  {h_ref_tf.dtype}')

## CELL 10 · Differentiable RT – Material Calibration

| Step | What | API |
|------|------|-----|
| 10.1 | Create `RadioMaterial` with `tf.Variable` properties | `RadioMaterial(name, relative_permittivity=tf.Variable(...))` |
| 10.2 | Redirect scene objects to trainable material | `obj.radio_material = name + '_train'` |
| 10.3 | NMSE loss over OFDM channel | `‖ĥ − h_ref‖² / ‖h_ref‖²` |
| 10.4 | Gradients via `tape.watched_variables()` | automatic |
| 10.5 | Clamp to physical range after each step | `check_mat()` |

In [ ]:
orig_params    = {}
original_mats  = {}
trainable_mats = {}
_train_suffix  = '_train'

print('Creating trainable RadioMaterial objects ...')
for mat_name, mat in list(scene.radio_materials.items()):
    if mat_name.endswith(_train_suffix): continue

    _used = False
    try:
        _used = mat.is_used
    except AttributeError:
        _used = any(
            getattr(obj, 'radio_material', None) is not None and
            getattr(getattr(obj, 'radio_material', None), 'name', '') == mat_name
            for obj in scene.objects.values())
    if not _used: continue

    key  = _match_itu(mat_name)
    _itu = _ITU_DB.get(key, _DEFAULT_MAT)
    eps0 = _itu[0]; sig0 = _itu[1]; S0 = _itu[2]
    try:
        v = mat.relative_permittivity
        eps0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    try:
        v = mat.conductivity
        sig0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    for a_ in ('scattering_coefficient','scattering_coeff'):
        if hasattr(mat, a_):
            try: S0 = float(getattr(mat,a_).numpy() if hasattr(getattr(mat,a_),'numpy') else getattr(mat,a_)); break
            except: pass

    orig_params[mat_name] = {'eps_r': eps0, 'sigma': sig0, 'S': S0}

    sn = mat_name.replace('/','_').replace(' ','_').replace('-','_')
    kw = dict(
        relative_permittivity = tf.Variable(eps0, dtype=tf.float32, name=f'{sn}_eps'),
        conductivity          = tf.Variable(sig0, dtype=tf.float32, name=f'{sn}_sig'),
    )
    try:
        new_mat = RadioMaterial(mat_name+_train_suffix,
                                scattering_coefficient=tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S'),
                                **kw)
    except TypeError:
        new_mat = RadioMaterial(mat_name+_train_suffix, **kw)
        try: new_mat.scattering_coefficient = tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S')
        except: pass

    scene.add(new_mat)
    original_mats[mat_name]  = mat
    trainable_mats[mat_name] = new_mat
    print(f'  {mat_name:<30} → {mat_name+_train_suffix}')
    print(f'    eps_r={eps0:.3f}  sigma={sig0:.4g}  S={S0:.2f}')

print()
n_redir = 0
for obj_name, obj in scene.objects.items():
    rm = getattr(obj, 'radio_material', None)
    if rm is None: continue
    orig_name = rm.name if hasattr(rm,'name') else str(rm)
    if orig_name in trainable_mats:
        try:
            obj.radio_material = orig_name + _train_suffix
            n_redir += 1
        except Exception as e:
            print(f'  WARNING [{obj_name}]: {e}')

print(f'Redirected {n_redir} scene objects to trainable materials.')
print(f'Trainable materials: {list(trainable_mats.keys())}')

In [ ]:
def nmse_loss(h_hat, h_ref):
    """NMSE = E[|h_hat - h_ref|²] / E[|h_ref|²]  (diff-rt eq.)"""
    h_hat = tf.cast(h_hat, h_ref.dtype)
    err   = tf.reduce_mean(tf.abs(h_hat - h_ref)**2)
    ref   = tf.reduce_mean(tf.abs(h_ref)**2) + 1e-30
    return err / ref

def check_mat(mat):
    """Clamp material properties to physical range (diff-rt check_mat)."""
    try:
        v = mat.relative_permittivity
        if hasattr(v, 'assign'):
            if _safe(v) < 1.0:  v.assign(tf.ones_like(v))
            if _safe(v) > 50.0: v.assign(50.0 * tf.ones_like(v))
    except: pass
    try:
        v = mat.conductivity
        if hasattr(v, 'assign') and _safe(v) < 0.0:
            v.assign(tf.zeros_like(v))
    except: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try:
                v = getattr(mat, a_)
                if hasattr(v,'assign'):
                    v.assign(tf.clip_by_value(v, 0.0, 1.0))
                break
            except: pass

print('NMSE loss + check_mat() defined.')
print(f'Loss signal: {"OFDM complex channel [rx, n_sub]" if _HAS_OFDM else "path power [rx] (scalar fallback)"}')

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=CALIB_LR)

history = {
    'step': [], 'nmse_db': [],
    'eps_r': {m: [] for m in trainable_mats},
    'sigma': {m: [] for m in trainable_mats},
    'S':     {m: [] for m in trainable_mats},
}

print(f'Material calibration – {CALIB_STEPS} steps')
print(f'  Adam LR={CALIB_LR}  depth={CALIB_DEPTH}  samp={CALIB_NUM_SAMP:,}')
print(f'  h_ref : {h_ref_tf.shape}  {h_ref_tf.dtype}')
print('-' * 70)

t0 = time.time()
for step in range(CALIB_STEPS):
    with tf.GradientTape() as tape:
        h_hat  = compute_h_freq(scene, num_samp=CALIB_NUM_SAMP, depth=CALIB_DEPTH)
        h_hat  = tf.cast(h_hat, h_ref_tf.dtype)
        min_rx = min(h_hat.shape[0], h_ref_tf.shape[0])
        loss   = nmse_loss(h_hat[:min_rx], h_ref_tf[:min_rx])

    watched = tape.watched_variables()
    grads   = tape.gradient(loss, watched,
                             unconnected_gradients=tf.UnconnectedGradients.ZERO)
    optimizer.apply_gradients(zip(grads, watched))

    for mat in trainable_mats.values():
        check_mat(mat)

    nmse_db = 10 * np.log10(float(loss.numpy()) + 1e-30)
    history['step'].append(step)
    history['nmse_db'].append(nmse_db)
    for mn, mat in trainable_mats.items():
        try: history['eps_r'][mn].append(_safe(mat.relative_permittivity))
        except: history['eps_r'][mn].append(float('nan'))
        try: history['sigma'][mn].append(_safe(mat.conductivity))
        except: history['sigma'][mn].append(float('nan'))
        for a_ in ('scattering_coefficient','scattering_coeff'):
            if hasattr(mat, a_):
                try: history['S'][mn].append(_safe(getattr(mat,a_))); break
                except: pass
        else: history['S'][mn].append(float('nan'))

    if step % 10 == 0 or step == CALIB_STEPS - 1:
        n0 = sum(1 for g in grads
                 if g is None or float(tf.reduce_sum(tf.abs(g)).numpy()) == 0)
        print(f'  step {step:4d}  NMSE={nmse_db:+6.1f} dB  '
              f'watched={len(watched)}  zero_grads={n0}  t={time.time()-t0:.0f}s')

print('-' * 70)
print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
n_mats   = len(trainable_mats)
colors   = plt.cm.tab10(np.linspace(0, 1, max(n_mats, 1)))
mat_list = list(trainable_mats.keys())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['step'], history['nmse_db'], 'k-', lw=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('NMSE (dB)')
axes[0].set_title('Calibration Loss (NMSE)'); axes[0].grid(True, alpha=0.4)

for i, mn in enumerate(mat_list):
    axes[1].plot(history['step'], history['eps_r'][mn],
                 label=mn.replace('_train',''), color=colors[i])
axes[1].set_xlabel('Step'); axes[1].set_ylabel('ε_r')
axes[1].set_title('Relative Permittivity'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.4)

for i, mn in enumerate(mat_list):
    axes[2].plot(history['step'], history['sigma'][mn],
                 label=mn.replace('_train',''), color=colors[i])
axes[2].set_xlabel('Step'); axes[2].set_ylabel('σ (S/m)')
axes[2].set_title('Conductivity'); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'calibration_convergence.png'), dpi=150)
plt.show()

In [ ]:
print(f'{"Material":<30} {"Param":<8} {"Initial":>10} {"Calibrated":>12} {"Delta":>8}')
print('=' * 75)

calib_results = {}
for mn, mat in trainable_mats.items():
    orig = orig_params.get(mn, {})
    row  = {}
    for pname, attr_candidates, key in [
        ('eps_r', ['relative_permittivity'], 'eps_r'),
        ('sigma', ['conductivity'],           'sigma'),
        ('S',     ['scattering_coefficient', 'scattering_coeff'], 'S'),
    ]:
        init_val = orig.get(key, float('nan'))
        cal_val  = float('nan')
        for a_ in attr_candidates:
            if hasattr(mat, a_):
                try: cal_val = _safe(getattr(mat, a_)); break
                except: pass
        delta = cal_val - init_val if not np.isnan(init_val) else float('nan')
        print(f'  {mn.replace(_train_suffix,""):<28} {pname:<8} '
              f'{init_val:>10.4f} {cal_val:>12.4f} {delta:>+8.4f}')
        row[pname] = {'initial': init_val, 'calibrated': cal_val}
    calib_results[mn] = row

out_json = os.path.join(OUTPUT_DIR, 'calibration_results.json')
with open(out_json, 'w') as f:
    json.dump(calib_results, f, indent=2)
print(f'\nCalibration JSON saved to {out_json}')

## CELL 11 · TX Orientation Optimization

| Parameter | Value |
|-----------|-------|
| Optimizer | **RMSprop** (diff-rt choice) |
| Loss | −E[log₂(1 + SNR × path_gain)] |
| Variable | `tx.orientation` as `tf.Variable` |
| Steps | `ORI_STEPS` |

In [ ]:
tx_name = list(scene.transmitters.keys())[0]
tx      = scene.transmitters[tx_name]

_ori_init = [0.0, 0.0, 0.0]
try: _ori_init = [_safe(tx.orientation[i]) for i in range(3)]
except: pass

tx.orientation = tf.Variable(_ori_init, dtype=tf.float32, name='tx_orientation')
print(f'TX "{tx_name}"  orientation = {_ori_init}  → tf.Variable')

def cm_capacity_loss(sc, cell_size=10.0, n_samp=ORI_NUM_SAMP):
    """Loss = −E[log₂(1 + SNR_scale × path_gain)]  (diff-rt Learning_Orientation)."""
    try:
        cm = sc.coverage_map(
            cm_cell_size        = cell_size,
            max_depth           = 3,
            num_samples         = n_samp,
            los                 = True,
            specular_reflection = True,
            diffuse_reflection  = False,
            refraction          = False,
            diffraction         = False,
        )
        pg = None
        for attr in ('path_gain', 'as_tensor'):
            if hasattr(cm, attr):
                val = getattr(cm, attr)
                pg  = val() if callable(val) else val
                break
        if pg is None: raise AttributeError('no path_gain')
        pg_flat  = tf.reshape(tf.cast(pg[0], tf.float32), [-1])
        capacity = tf.reduce_mean(
            tf.math.log(1.0 + SNR_SCALE * pg_flat) / tf.math.log(2.0))
        return -capacity, cm
    except Exception as e:
        print(f'  cm_capacity_loss error: {e}')
        return tf.constant(0.0), None

print(f'SNR_SCALE = {SNR_SCALE:.2e}')

In [ ]:
ori_optimizer = tf.keras.optimizers.RMSprop(learning_rate=ORI_LR)
ori_history   = {'step': [], 'rate_bit': [], 'orientation': []}

print(f'TX orientation optimization – {ORI_STEPS} steps  RMSprop LR={ORI_LR}')
print('-' * 60)

cm_before_np = None
t0 = time.time()
for step in range(ORI_STEPS):
    with tf.GradientTape() as tape:
        loss_val, cm_opt = cm_capacity_loss(scene, cell_size=10.0, n_samp=ORI_NUM_SAMP)

    if step == 0 and cm_opt is not None:
        cm_before_np = _cm_to_numpy(cm_opt)

    grads    = tape.gradient(loss_val, tape.watched_variables())
    valid_gv = [(g, v) for g, v in zip(grads, tape.watched_variables()) if g is not None]
    if valid_gv: ori_optimizer.apply_gradients(valid_gv)

    rate = float(-loss_val.numpy())
    ori  = list(tx.orientation.numpy())
    ori_history['step'].append(step)
    ori_history['rate_bit'].append(rate)
    ori_history['orientation'].append(ori)
    print(f'  step {step:3d}  rate={rate:.4f} bit  '
          f'ori=[{", ".join(f"{o:.3f}" for o in ori)}]  t={time.time()-t0:.0f}s', end='\r')

print()
print('-' * 60)
ori_final = list(tx.orientation.numpy())
print(f'Initial orientation  : {_ori_init}')
print(f'Optimized orientation: {[round(o, 4) for o in ori_final]}')
rate_initial = ori_history['rate_bit'][0]  if ori_history['rate_bit'] else 0
rate_final   = ori_history['rate_bit'][-1] if ori_history['rate_bit'] else 0
print(f'Rate improvement     : {rate_initial:.4f} → {rate_final:.4f} bit  (+{rate_final-rate_initial:.4f})')

## CELL 12 · Post-Calibration Analysis

In [ ]:
print('Post-calibration path computation ...')
print(f'  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

paths_cal = scene.compute_paths(
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_PS,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = False,
)
print('Done.')

a_np   = _to_numpy(paths_cal.a)
tau_np = _to_numpy(paths_cal.tau)
if a_np.ndim == 6: a_np = a_np[0]

n_rx    = a_np.shape[0]
power   = np.sum(np.abs(a_np)**2, axis=tuple(range(1, a_np.ndim)))
pg_cal  = power
pg_cal_db = 10 * np.log10(pg_cal + 1e-30)

print(f'Post-calibration path gain at {n_rx} receivers:')
print(f'  mean={pg_cal_db.mean():.1f} dB  min={pg_cal_db.min():.1f} dB  max={pg_cal_db.max():.1f} dB')

In [ ]:
records = []
for i, rx in enumerate(receivers[:n_rx]):
    x   = _safe(rx.position[0])
    y   = _safe(rx.position[1])
    z   = _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    pg_pre  = float(pg_at_rx_pre[i]) if i < len(pg_at_rx_pre) else float('nan')
    pg_post = float(pg_cal_db[i])    if i < len(pg_cal_db)    else float('nan')
    records.append({
        'receiver'    : rx.name,
        'lon'         : round(lon, 6),
        'lat'         : round(lat, 6),
        'x_m'         : round(x,   2),
        'y_m'         : round(y,   2),
        'z_m'         : round(z,   3),
        'pg_pre_db'   : round(pg_pre,  2),
        'pg_post_db'  : round(pg_post, 2),
        'delta_pg_db' : round(pg_post - pg_pre, 2) if not np.isnan(pg_pre) else float('nan'),
    })

df_out = pd.DataFrame(records)
out_csv = os.path.join(OUTPUT_DIR, 'receiver_results_calibrated.csv')
df_out.to_csv(out_csv, index=False)
print(f'Saved {len(df_out)} receivers to {out_csv}')
print(df_out.head(10).to_string(index=False))

In [ ]:
print('Computing final calibrated coverage map ...')
cm_final    = scene.coverage_map(
    cm_cell_size        = GRID_SIZE_M,
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_CM,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = False,
)
cm_final_np    = _cm_to_numpy(cm_final)
pg_pre_db_2d   = 10 * np.log10(cm_pre_np[0]   + 1e-30)
pg_final_db_2d = 10 * np.log10(cm_final_np[0] + 1e-30)

vmin = min(np.nanpercentile(pg_pre_db_2d, 5),  np.nanpercentile(pg_final_db_2d, 5))
vmax = max(np.nanpercentile(pg_pre_db_2d, 99), np.nanpercentile(pg_final_db_2d, 99))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, data, title in [
    (axes[0], pg_pre_db_2d,   'Before Calibration (ITU defaults)'),
    (axes[1], pg_final_db_2d, 'After Calibration (diff-rt)'),
    (axes[2], pg_final_db_2d - pg_pre_db_2d, 'Δ Path Gain (After − Before)'),
]:
    if 'Δ' in title:
        im = ax.imshow(data, origin='lower', cmap='RdYlGn', vmin=-10, vmax=10)
    else:
        im = ax.imshow(data, origin='lower', cmap='jet', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Path Gain (dB)')
    ax.set_title(title); ax.set_xlabel('X cells'); ax.set_ylabel('Y cells')

plt.suptitle('Coverage Map: Before vs After Differentiable RT Calibration', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nAll results saved to:', OUTPUT_DIR)